In [ ]:
import pandas as pd
try:
    df = pd.read_csv("combined_data.csv")
except UnicodeDecodeError:
    df = pd.read_csv("combined_data.csv", encoding="latin-1")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
print("Shape (rows, columns):", df.shape)
print("\nColumns:", list(df.columns))
print("\nPehli 5 rows:")
df.head()

Shape (rows, columns): (83448, 2)

Columns: ['label', 'text']

Pehli 5 rows:


,label,text
0,1,ounce feather bowl hummingbird opec moment ala...
1,1,wulvob get your medircations online qnb ikud v...
2,0,computer connection from cnn com wednesday es...
3,1,university degree obtain a prosperous future m...
4,0,thanks for all your answers guys i know i shou...


In [ ]:
print("Missing values:")
print(df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())


Missing values:
label    0
text     0
dtype: int64

Duplicate rows: 0


In [ ]:
print("\nSpam(1) vs Ham(0) count:")
print(df['label'].value_counts())


Spam(1) vs Ham(0) count:
label
1    43910
0    39538
Name: count, dtype: int64


In [ ]:
# 1. Poora shape dobara confirm karo
print("Shape:", df.shape)


# 2. Har email mein kitne words hain
df['word_count'] = df['text'].str.split().str.len()

print("\nWord count statistics:")
print(df['word_count'].describe())


# 3. Bahut chhote emails (5 words se kam)
short_emails = df[df['word_count'] < 5]
print("\nBahut chhote emails (5 words se kam):", len(short_emails))


# 4. Kuch examples dekho
print("\n--- 2 SPAM examples ---")
for email in df[df['label'] == 1]['text'].head(2):
    print(email[:300], "\n")

print("--- 2 HAM examples ---")
for email in df[df['label'] == 0]['text'].head(2):
    print(email[:300], "\n")

Shape: (83448, 2)

Word count statistics:
count     83448.000000
mean        282.811775
std         724.818152
min           1.000000
25%          80.000000
50%         152.000000
75%         312.000000
max      101984.000000
Name: word_count, dtype: float64

Bahut chhote emails (5 words se kam): 297

--- 2 SPAM examples ---
ounce feather bowl hummingbird opec moment alabaster valkyrie dyad bread flack desperate iambic hadron heft quell yoghurt bunkmate divert afterimage 

wulvob get your medircations online qnb ikud viagra escapenumber escapenumber levitra escapenumber escapenumber cialis escapenumber escapenumber imitrex escapenumber escapenumber flonax escapenumber escapenumber ultram escapenumber escapenumber vioxx escapenumber escapenumber ambien escapenumber esc 

--- 2 HAM examples ---
 computer connection from cnn com wednesday escapenumber may escapenumber escapenumber escapenumber escapenumber pm edt in this report next generation toys read brain waves google expands personal

In [ ]:
short_emails = df[df['word_count'] < 5]


print("Short emails mein spam vs ham:")
print(short_emails['label'].value_counts())


print("\nKuch examples:")
print(short_emails[['label', 'text']].head(15))

Short emails mein spam vs ham:
label
1    221
0     76
Name: count, dtype: int64

Kuch examples:
      label                             text
478       1                  column topphpbb
1051      1                hi !\nxaawa tel\n
1325      1                 hola !\nhej da\n
1974      1                       buy it !\n
2117      1                           maxine
2176      1  hilloescapenumber fescapenumber
2235      0                           done .
2380      1                         hassan\n
2966      1                 laptops to buy !
3209      1     tlxjwe quama aamlahu gopohlu
3382      1      split content escapenumber 
3577      1        jgvnth efqw lroym rbqfhcq
3716      0       please see attached letter
3808      0                   proposed sun\n
3910      1                            degli


In [ ]:
before = len(df)


# 5 ya usse zyada words wale emails hi rakho
df = df[df['word_count'] >= 5]


# Row numbers dobara 0 se shuru karo
df = df.reset_index(drop=True)


after = len(df)


print("Pehle rows:", before)
print("Ab rows:", after)
print("Hataye gaye:", before - after)


print("\nSpam vs Ham ab:")
print(df['label'].value_counts())

Pehle rows: 83448
Ab rows: 83151
Hataye gaye: 297

Spam vs Ham ab:
label
1    43689
0    39462
Name: count, dtype: int64


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd


# 3 chhote emails
emails = [
    "free prize click now",
    "meeting is at 10",
    "free meeting now",
]


# TF-IDF vectorizer banao
vectorizer = TfidfVectorizer()


# Emails se seekho aur unhe numbers mein badlo
numbers = vectorizer.fit_transform(emails)


# Vocabulary (saare unique words)
print("Vocabulary:")
print(vectorizer.get_feature_names_out())


# Numbers ko ek table mein dekho
table = pd.DataFrame(
    numbers.toarray(),
    columns=vectorizer.get_feature_names_out(),
)

table.round(2)

Vocabulary:
['10' 'at' 'click' 'free' 'is' 'meeting' 'now' 'prize']


,10,at,click,free,is,meeting,now,prize
0,0.00,0.00,0.56,0.43,0.00,0.00,0.43,0.56
1,0.53,0.53,0.00,0.00,0.53,0.40,0.00,0.00
2,0.00,0.00,0.00,0.58,0.00,0.58,0.58,0.00


In [ ]:
from sklearn.model_selection import train_test_split


# X = input (email text), y = answer (spam ya ham)
X = df['text']
y = df['label']


# 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)


print("Training emails:", len(X_train))
print("Testing emails:", len(X_test))


print("\nTraining mein spam/ham %:")
print(y_train.value_counts(normalize=True) * 100)


print("\nTesting mein spam/ham %:")
print(y_test.value_counts(normalize=True) * 100)

Training emails: 66520
Testing emails: 16631

Training mein spam/ham %:
label
1    52.542093
0    47.457907
Name: proportion, dtype: float64

Testing mein spam/ham %:
label
1    52.540437
0    47.459563
Name: proportion, dtype: float64


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer


# TF-IDF vectorizer banao
vectorizer = TfidfVectorizer(max_features=20000)


# Training data se seekho + numbers mein badlo
X_train_tfidf = vectorizer.fit_transform(X_train)


# Testing data ko sirf numbers mein badlo (seekhna nahi)
X_test_tfidf = vectorizer.transform(X_test)


print("Train shape:", X_train_tfidf.shape)
print("Test shape:", X_test_tfidf.shape)


# Vocabulary ke kuch words dekho
words = vectorizer.get_feature_names_out()

print("\nVocabulary size:", len(words))
print("Kuch words:", words[:15])
print("Kuch aur words:", words[10000:10015])

Train shape: (66520, 20000)
Test shape: (16631, 20000)

Vocabulary size: 20000
Kuch words: ['00' '000' '0000' '000000' '000080' '001' '0011' '002' '0031' '005' '01'
 '010' '0100' '011' '012']
Kuch aur words: ['kuleuven' 'kumar' 'kunden' 'kurs' 'kurt' 'kurz' 'kw' 'ky' 'kyle' 'kysa'
 'la' 'lab' 'label' 'labeled' 'labelled']


In [ ]:
import numpy as np


# Training ki pehli email
i = 0
row = X_train_tfidf[i]


# Is email mein jo words hain, unke column numbers aur scores
indices = row.indices
scores = row.data


# Sabse bade 8 scores (bade se chhote order mein)
top = np.argsort(scores)[::-1][:8]


print("Email ka text (pehle 200 characters):")
print(X_train.iloc[i][:200])

print("\nLabel:", y_train.iloc[i], "(1 = spam, 0 = ham)")

print("\nTop words aur unka TF-IDF score:")
for j in top:
    print(words[indices[j]], "->", round(scores[j], 3))

print("\nIs email mein kitne alag words hain:", row.nnz)

Email ka text (pehle 200 characters):
maybe this chkescapenumber function x chr deparse substitute x e parse text chr structure exists chr parent env environment is name length e escapenumber is name e escapenumber chkescapenumber escapen

Label: 0 (1 = spam, 0 = ham)

Top words aur unka TF-IDF score:
function -> 0.355
exists -> 0.333
escapenumber -> 0.305
argument -> 0.294
false -> 0.285
the -> 0.198
inside -> 0.146
talbot -> 0.146

Is email mein kitne alag words hain: 258


In [ ]:
from sklearn.naive_bayes import MultinomialNB


# Model banao
model = MultinomialNB()


# Training data se seekhao
model.fit(X_train_tfidf, y_train)


print("Model train ho gaya!")


# Test data par pehla quick check
predictions = model.predict(X_test_tfidf)

accuracy = (predictions == y_test).mean()

print("Test accuracy:", round(accuracy * 100, 2), "%")

Model train ho gaya!
Test accuracy: 97.35 %


In [ ]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

import pandas as pd


# Model se test data ki predictions
predictions = model.predict(X_test_tfidf)


# 1. Accuracy
print("Accuracy:", round(accuracy_score(y_test, predictions) * 100, 2), "%")


# 2. Precision, Recall, F1
print("\nClassification Report:")
print(classification_report(y_test, predictions, target_names=["HAM", "SPAM"]))


# 3. Confusion Matrix
cm = confusion_matrix(y_test, predictions)

cm_table = pd.DataFrame(
    cm,
    index=["Asli HAM", "Asli SPAM"],
    columns=["Predicted HAM", "Predicted SPAM"],
)

print("Confusion Matrix:")
print(cm_table)

Accuracy: 97.35 %

Classification Report:
              precision    recall  f1-score   support

         HAM       0.97      0.98      0.97      7893
        SPAM       0.98      0.97      0.97      8738

    accuracy                           0.97     16631
   macro avg       0.97      0.97      0.97     16631
weighted avg       0.97      0.97      0.97     16631

Confusion Matrix:
           Predicted HAM  Predicted SPAM
Asli HAM            7731             162
Asli SPAM            278            8460


In [ ]:
import os
import joblib
import sklearn


# "model" naam ka folder banao (agar pehle se hai to koi dikkat nahi)
os.makedirs("model", exist_ok=True)


# Dono ko save karo
joblib.dump(model, "model/spam_model.pkl")
joblib.dump(vectorizer, "model/tfidf_vectorizer.pkl")


print("Saved files:", os.listdir("model"))
print("scikit-learn version:", sklearn.__version__)

Saved files: ['tfidf_vectorizer.pkl', 'spam_model.pkl']
scikit-learn version: 1.6.1


In [ ]:
# Save ki hui files wapas load karo
loaded_model = joblib.load("model/spam_model.pkl")
loaded_vectorizer = joblib.load("model/tfidf_vectorizer.pkl")


# Test emails dobara numbers mein badlo aur predict karo
test_numbers = loaded_vectorizer.transform(X_test)
test_predictions = loaded_model.predict(test_numbers)


accuracy = (test_predictions == y_test).mean()

print("Loaded model ki accuracy:", round(accuracy * 100, 2), "%")

Loaded model ki accuracy: 97.35 %


In [ ]:
!zip -r model.zip model

from google.colab import files
files.download("model.zip")

  adding: model/ (stored 0%)
  adding: model/tfidf_vectorizer.pkl (deflated 69%)
  adding: model/spam_model.pkl (deflated 12%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import re


def clean_text(text):

    # 1. Lowercase
    text = text.lower()

    # 2. Har number ki jagah "escapenumber"
    text = re.sub(r"\d+", " escapenumber ", text)

    # 3. Letters aur spaces ke alawa sab kuch space se badlo
    text = re.sub(r"[^a-z\s]", " ", text)

    # 4. Extra spaces hatao
    text = re.sub(r"\s+", " ", text).strip()

    return text


def predict_email(email):

    # Naye email ko dataset ki style mein badlo
    cleaned = clean_text(email)

    # SAME vectorizer se numbers banao (sirf transform, fit nahi)
    numbers = loaded_vectorizer.transform([cleaned])

    # Prediction aur probability
    prediction = loaded_model.predict(numbers)[0]
    probability = loaded_model.predict_proba(numbers)[0]

    spam_chance = probability[1] * 100

    print("Cleaned text:", cleaned)

    if prediction == 1:
        print("Prediction: 🚨 SPAM")
    else:
        print("Prediction: ✅ NOT SPAM")

    print("Spam hone ki probability:", round(spam_chance, 1), "%")
    print("-" * 50)


# Naye emails (training data mein nahi the)
predict_email("Congratulations! You have won a free iPhone. Click here to claim your prize.")

predict_email("Hi, please send me the project report before 5 PM.")

Cleaned text: congratulations you have won a free iphone click here to claim your prize
Prediction: 🚨 SPAM
Spam hone ki probability: 90.7 %
--------------------------------------------------
Cleaned text: hi please send me the project report before escapenumber pm
Prediction: ✅ NOT SPAM
Spam hone ki probability: 12.1 %
--------------------------------------------------


In [ ]:
test_emails = [

    # Aasan spam
    "URGENT! You have been selected to receive a $5000 cash prize. Reply now to claim!",

    # Aasan ham
    "Hi team, the meeting is rescheduled to 3 PM tomorrow. Please update your calendar.",

    # College wala ham
    "Dear students, please submit the ML assignment before Friday. No extensions will be given.",

    # Spam jo thoda professional lagta hai
    "Dear customer, your bank account has been suspended. Verify your details immediately to avoid closure.",

    # Ham jo spam jaisa dikhta hai (offer/sale words)
    "Hi, the college is offering a free workshop on machine learning next week. Register here.",

    # Hinglish (model ne kabhi nahi dekha)
    "Bhai kal ka assignment bhej dena, mujhe submit karna hai.",
]


for email in test_emails:
    print("EMAIL:", email)
    predict_email(email)

EMAIL: URGENT! You have been selected to receive a $5000 cash prize. Reply now to claim!
Cleaned text: urgent you have been selected to receive a escapenumber cash prize reply now to claim
Prediction: 🚨 SPAM
Spam hone ki probability: 93.9 %
--------------------------------------------------
EMAIL: Hi team, the meeting is rescheduled to 3 PM tomorrow. Please update your calendar.
Cleaned text: hi team the meeting is rescheduled to escapenumber pm tomorrow please update your calendar
Prediction: ✅ NOT SPAM
Spam hone ki probability: 4.5 %
--------------------------------------------------
EMAIL: Dear students, please submit the ML assignment before Friday. No extensions will be given.
Cleaned text: dear students please submit the ml assignment before friday no extensions will be given
Prediction: ✅ NOT SPAM
Spam hone ki probability: 10.7 %
--------------------------------------------------
EMAIL: Dear customer, your bank account has been suspended. Verify your details immediately to avoid

In [ ]:
import sklearn

print(sklearn.__version__)

1.6.1
